In [1]:
import torch
print("torch version:", torch.__version__)
print("CUDA in torch:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {props.name}  {props.total_memory/1e9:.1f} GB  CC {props.major}.{props.minor}")

torch version: 2.6.0+cu124
CUDA in torch: 12.4
CUDA available: True
Device: Quadro P5000  17.1 GB  CC 6.1


In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier
from interpret.glassbox import ExplainableBoostingClassifier
from tqdm.auto import tqdm
from scipy.stats import spearmanr
from itertools import combinations
from time import perf_counter
import warnings
warnings.filterwarnings("ignore")

import torch
from tabicl import TabICLClassifier

np.random.seed(4)


# ==============================================================
# INLINED UTILS (from recgam.utils)
# ==============================================================
def calculate_global_explanation_multiplicity(importances_matrix, k=5):
    """Global explanation multiplicity metrics for an (n_models, n_features) importance matrix."""
    n_models, n_features = importances_matrix.shape

    # 1. Spearman's rank correlation, pairwise
    spearman_corrs = []
    for model_a, model_b in combinations(range(n_models), 2):
        corr, _ = spearmanr(importances_matrix[model_a], importances_matrix[model_b])
        spearman_corrs.append(corr)
    mean_spearman = np.mean(spearman_corrs)

    # 2. Top-K overlap (mean Jaccard index)
    top_k_indices = [set(np.argsort(importances_matrix[i])[-k:]) for i in range(n_models)]
    jaccard_scores = []
    for set_a, set_b in combinations(top_k_indices, 2):
        intersection = len(set_a.intersection(set_b))
        union = len(set_a.union(set_b))
        jaccard_scores.append(intersection / union)
    mean_jaccard = np.mean(jaccard_scores)

    # 3. Per-feature std
    std_per_feature = np.std(importances_matrix, axis=0)

    return {
        "mean_spearman_correlation": mean_spearman,
        "mean_top_k_jaccard": mean_jaccard,
        "std_per_feature": std_per_feature,
    }


def calculate_disagreement(preds_class):
    """Mean pairwise class disagreement across models."""
    _, num_models = preds_class.shape
    disagreements = []
    for model_a, model_b in combinations(range(num_models), 2):
        pair_disagreement = np.mean(preds_class[:, model_a] != preds_class[:, model_b])
        disagreements.append(pair_disagreement)
    return np.mean(disagreements), disagreements


def calculate_ambiguity(preds_proba):
    """Per-sample variance and range of predicted probabilities across models."""
    sample_variances = np.var(preds_proba, axis=1)
    sample_ranges = np.max(preds_proba, axis=1) - np.min(preds_proba, axis=1)
    return sample_variances, sample_ranges


# --------------------------------------------------------------
# Runtime controls
# --------------------------------------------------------------
USE_CALIBRATED  = False
TABICL_DEVICE   = "cuda"
TABICL_EST      = 4
TABICL_BATCH    = 4
TABICL_OOF_SPL  = 3
XGB_OOF_SPL     = 5
N_SEEDS         = 5
EBM_MAX_ROUNDS  = 50

print("=" * 60)
print("GPU check")
print("=" * 60)
print(f"  torch: {torch.__version__}   cuda: {torch.version.cuda}")
print(f"  cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  device: {props.name}  {props.total_memory/1e9:.1f} GB  CC {props.major}.{props.minor}")
else:
    print("  ⚠ no CUDA — TabICL will run on CPU")
    TABICL_DEVICE = "cpu"


# ==============================================================
# 1. LOAD ADULT
# ==============================================================
NPZ_PATH = "data/adult/adult_tabpfn_data.npz"

with np.load(NPZ_PATH, allow_pickle=False) as data:
    columns  = data["feature_columns"].tolist()
    _x_train = data["df_x_train"]
    _x_val   = data["df_x_val"]
    _x_test  = data["df_x_test"]
    _y_train = data["y_train"].copy()
    _y_val   = data["y_val"].copy()
    _y_test  = data["y_test"].copy()

X_train = pd.concat(
    [pd.DataFrame(_x_train, columns=columns),
     pd.DataFrame(_x_val,   columns=columns)],
    ignore_index=True,
)
X_test  = pd.DataFrame(_x_test, columns=columns).reset_index(drop=True)
y_train = np.concatenate([np.asarray(_y_train), np.asarray(_y_val)])
y_test  = np.asarray(_y_test)

# Insert right after loading:
X_train_full = X_train.copy()
y_train_full = y_train.copy()
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), size=7500, replace=False)
X_train = X_train.iloc[idx].reset_index(drop=True)
y_train = y_train[idx]
# Keep X_test at full 9.7k — evaluation remains comparable

print(f"\nTrain: {X_train.shape}   Test: {X_test.shape}   "
      f"base rate train={y_train.mean():.4f} test={y_test.mean():.4f}")



# ==============================================================
# 2. TEACHER A: TabICL on GPU
# ==============================================================
def make_tabicl():
    return TabICLClassifier(
        n_estimators=TABICL_EST,
        batch_size=TABICL_BATCH,
        device=TABICL_DEVICE,
        use_amp="auto",
        use_fa3="auto",
        random_state=42,
    )

print(f"\n[1/4] TabICL teacher (GPU, n_estimators={TABICL_EST})...")
t0 = perf_counter()
tabicl_teacher = make_tabicl()
tabicl_teacher.fit(X_train, y_train)
p_test_tabicl = tabicl_teacher.predict_proba(X_test)[:, 1]
print(f"    done in {perf_counter()-t0:.1f}s  |  "
      f"test AUC = {roc_auc_score(y_test, p_test_tabicl):.4f}")


# ==============================================================
# 3. TabICL OOF
# ==============================================================
def oof_probs_tabicl(X, y, n_splits=TABICL_OOF_SPL):
    p_oof = np.zeros(len(X))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for tr, va in tqdm(skf.split(X, y), desc="TabICL OOF",
                       leave=False, dynamic_ncols=True):
        m = make_tabicl()
        m.fit(X.iloc[tr], y[tr])
        p_oof[va] = m.predict_proba(X.iloc[va])[:, 1]
    return p_oof

print(f"\n[2/4] TabICL OOF ({TABICL_OOF_SPL} folds, GPU)...")
t0 = perf_counter()
p_train_tabicl = np.clip(oof_probs_tabicl(X_train, y_train), 1e-6, 1 - 1e-6)
print(f"    done in {perf_counter()-t0:.1f}s  |  "
      f"OOF train AUC = {roc_auc_score(y_train, p_train_tabicl):.4f}")

if USE_CALIBRATED:
    iso_tabicl = IsotonicRegression(out_of_bounds="clip").fit(p_train_tabicl, y_train)
    p_train_tabicl_cal = iso_tabicl.predict(p_train_tabicl)


# ==============================================================
# 4. TEACHER B: XGBoost OOF (CPU)
# ==============================================================
def oof_probs(model, X, y, n_splits=XGB_OOF_SPL):
    p_oof = np.zeros(len(X))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for tr, va in tqdm(skf.split(X, y), desc="XGB OOF",
                       leave=False, dynamic_ncols=True):
        m = model.__class__(**model.get_params())
        m.fit(X.iloc[tr], y[tr])
        p_oof[va] = m.predict_proba(X.iloc[va])[:, 1]
    return p_oof

xgb_params = dict(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42,
)
print(f"\n[3/4] XGBoost teacher + OOF ({XGB_OOF_SPL} folds, CPU)...")
t0 = perf_counter()
xgb_teacher = XGBClassifier(**xgb_params).fit(X_train, y_train)
p_train_xgb = oof_probs(XGBClassifier(**xgb_params), X_train, y_train)
p_test_xgb  = xgb_teacher.predict_proba(X_test)[:, 1]
print(f"    done in {perf_counter()-t0:.1f}s  |  "
      f"test AUC = {roc_auc_score(y_test, p_test_xgb):.4f}")

if USE_CALIBRATED:
    iso_xgb = IsotonicRegression(out_of_bounds="clip").fit(p_train_xgb, y_train)
    p_train_xgb_cal = iso_xgb.predict(p_train_xgb)


# ==============================================================
# 5. Teacher diagnostics
# ==============================================================
def teacher_diagnostics(name, p_train, p_test):
    auc_tr = roc_auc_score(y_train, p_train)
    auc_te = roc_auc_score(y_test,  p_test)
    gap = auc_tr - auc_te
    flag = f"  ⚠ gap {gap:+.3f}" if gap > 0.03 else ""
    print(f"  {name:10s}  train={auc_tr:.4f}  test={auc_te:.4f}  "
          f"LL={log_loss(y_test, p_test):.4f}  "
          f"Brier={brier_score_loss(y_test, p_test):.4f}{flag}")

print("\nTeacher diagnostics:")
teacher_diagnostics("TabICL", p_train_tabicl, p_test_tabicl)
teacher_diagnostics("XGBoost", p_train_xgb,  p_test_xgb)

rho = spearmanr(p_test_tabicl, p_test_xgb).statistic
print(f"\nTeacher agreement (Spearman): ρ = {rho:.4f}")


# ==============================================================
# 6. Helpers
# ==============================================================
def expand_soft_targets(x, target):
    p = np.clip(np.asarray(target, dtype=float), 0.0, 1.0)
    x_doubled = pd.concat([x, x], axis=0, ignore_index=True)
    y_doubled = np.concatenate([np.ones(len(x), dtype=int), np.zeros(len(x), dtype=int)])
    w_doubled = np.concatenate([p, 1.0 - p])
    keep = w_doubled > 1e-8
    return x_doubled.loc[keep].reset_index(drop=True), y_doubled[keep], w_doubled[keep]


def multiplicity_probe(fit_fn, X_tr, target_tr, X_va, y_va,
                       n_seeds=N_SEEDS, desc="seeds"):
    preds_proba_list, preds_class_list, importances_list = [], [], []
    for seed in tqdm(range(n_seeds), desc=desc, leave=False, dynamic_ncols=True):
        clf = fit_fn(X_tr, target_tr, seed)
        preds_proba_list.append(clf.predict_proba(X_va)[:, 1])
        preds_class_list.append(clf.predict(X_va))
        importances_list.append(clf.explain_global().data()["scores"])

    preds_proba = np.column_stack(preds_proba_list)
    preds_class = np.column_stack(preds_class_list)
    return {
        "disagreement":     float(calculate_disagreement(preds_class)[0]),
        "ambiguity":        float(calculate_ambiguity(preds_proba)[0].mean()),
        "explanation_corr": float(
            calculate_global_explanation_multiplicity(
                np.array(importances_list)
            )["mean_spearman_correlation"]
        ),
    }


def fit_ebm_hard(X, target, seed):
    return ExplainableBoostingClassifier(
        interactions="2x", max_rounds=EBM_MAX_ROUNDS,
        random_state=seed, n_jobs=-1,
    ).fit(X, target)


def fit_ebm_soft(X, target, seed):
    Xs, ys, ws = expand_soft_targets(X, target)
    return ExplainableBoostingClassifier(
        interactions="2x", max_rounds=EBM_MAX_ROUNDS,
        random_state=seed, n_jobs=-1,
    ).fit(Xs, ys, sample_weight=ws)


# ==============================================================
# 7. CONFIGS
# ==============================================================
CONFIGS = [
    ("EBM / binary",       fit_ebm_hard, "y",        "ebm"),
    ("EBM / TabICL probs", fit_ebm_soft, "p_tabicl", "ebm"),
    ("EBM / XGB probs",    fit_ebm_soft, "p_xgb",    "ebm"),
]
TARGETS = {
    "y":        y_train,
    "p_tabicl": p_train_tabicl,
    "p_xgb":    p_train_xgb,
}

if USE_CALIBRATED:
    CONFIGS += [
        ("EBM / TabICL probs (calib)", fit_ebm_soft, "p_tabicl_cal", "ebm"),
        ("EBM / XGB probs (calib)",    fit_ebm_soft, "p_xgb_cal",    "ebm"),
    ]
    TARGETS["p_tabicl_cal"] = p_train_tabicl_cal
    TARGETS["p_xgb_cal"]    = p_train_xgb_cal

print(f"\n[4/4] EBM students  ({len(CONFIGS)} configs × {N_SEEDS} seeds × 2 folds)")


# ==============================================================
# 8. Multiplicity probe
# ==============================================================
kf = StratifiedKFold(n_splits=2, shuffle=True, random_state=34)
multiplicity_rows = []

t_mult_start = perf_counter()
for label, fit_fn, target_key, family in tqdm(CONFIGS, desc="multiplicity", dynamic_ncols=True):
    target_full = TARGETS[target_key]
    fold_metrics = []
    for fold, (tr_idx, va_idx) in enumerate(
        tqdm(kf.split(X_train, y_train), desc=f"  {label}",
             leave=False, total=kf.get_n_splits(), dynamic_ncols=True)
    ):
        m = multiplicity_probe(
            fit_fn=fit_fn,
            X_tr=X_train.iloc[tr_idx].reset_index(drop=True),
            target_tr=target_full[tr_idx],
            X_va=X_train.iloc[va_idx].reset_index(drop=True),
            y_va=y_train[va_idx],
            n_seeds=N_SEEDS,
            desc=f"    fold {fold+1}",
        )
        fold_metrics.append(m)

    multiplicity_rows.append({
        "model":            label,
        "disagreement":     np.mean([m["disagreement"]     for m in fold_metrics]),
        "ambiguity":        np.mean([m["ambiguity"]        for m in fold_metrics]),
        "explanation_corr": np.nanmean([m["explanation_corr"] for m in fold_metrics]),
    })
print(f"    multiplicity done in {perf_counter()-t_mult_start:.1f}s")

multiplicity_df = pd.DataFrame(multiplicity_rows)


# ==============================================================
# 9. Performance on test
# ==============================================================
def ece_score(y_true, p, n_bins=10):
    frac_pos, mean_pred = calibration_curve(y_true, p, n_bins=n_bins)
    return float(np.mean(np.abs(frac_pos - mean_pred)))


perf_rows = [
    {"model": "TabICL (teacher)",
     "AUC": roc_auc_score(y_test, p_test_tabicl), "AUC_std": 0.0,
     "LogLoss": log_loss(y_test, p_test_tabicl),
     "Brier":   brier_score_loss(y_test, p_test_tabicl),
     "ECE":     ece_score(y_test, p_test_tabicl)},
    {"model": "XGBoost (teacher)",
     "AUC": roc_auc_score(y_test, p_test_xgb), "AUC_std": 0.0,
     "LogLoss": log_loss(y_test, p_test_xgb),
     "Brier":   brier_score_loss(y_test, p_test_xgb),
     "ECE":     ece_score(y_test, p_test_xgb)},
]

t_perf_start = perf_counter()
for label, fit_fn, target_key, family in tqdm(CONFIGS, desc="performance", dynamic_ncols=True):
    target_full = TARGETS[target_key]
    seed_metrics = []
    for seed in tqdm(range(N_SEEDS), desc=f"  {label}",
                     leave=False, dynamic_ncols=True):
        clf = fit_fn(X_train, target_full, seed)
        proba = clf.predict_proba(X_test)[:, 1]
        seed_metrics.append({
            "AUC":     roc_auc_score(y_test, proba),
            "LogLoss": log_loss(y_test, proba),
            "Brier":   brier_score_loss(y_test, proba),
            "ECE":     ece_score(y_test, proba),
        })

    perf_rows.append({
        "model":   label,
        "AUC":     np.mean([m["AUC"]     for m in seed_metrics]),
        "AUC_std": np.std( [m["AUC"]     for m in seed_metrics]),
        "LogLoss": np.mean([m["LogLoss"] for m in seed_metrics]),
        "Brier":   np.mean([m["Brier"]   for m in seed_metrics]),
        "ECE":     np.mean([m["ECE"]     for m in seed_metrics]),
    })
print(f"    performance done in {perf_counter()-t_perf_start:.1f}s")

performance_df = pd.DataFrame(perf_rows)


# ==============================================================
# 10. Combine + print
# ==============================================================
combined = performance_df.merge(
    multiplicity_df[["model", "disagreement", "ambiguity", "explanation_corr"]],
    on="model", how="left",
)

print("\n" + "=" * 100)
print(f"COMBINED  [USE_CALIBRATED={USE_CALIBRATED}, "
      f"TABICL_EST={TABICL_EST}, TABICL_OOF_SPL={TABICL_OOF_SPL}, "
      f"N_SEEDS={N_SEEDS}, EBM_MAX_ROUNDS={EBM_MAX_ROUNDS}]")
print("=" * 100)
print(combined.to_string(index=False, float_format="{:.4f}".format))

GPU check
  torch: 2.6.0+cu124   cuda: 12.4
  cuda available: True
  device: Quadro P5000  17.1 GB  CC 6.1

Train: (7500, 14)   Test: (9769, 14)   base rate train=0.2381 test=0.2393

[1/4] TabICL teacher (GPU, n_estimators=4)...
    done in 83.2s  |  test AUC = 0.9224

[2/4] TabICL OOF (3 folds, GPU)...


TabICL OOF: 0it [00:00, ?it/s]

    done in 80.8s  |  OOF train AUC = 0.9138

[3/4] XGBoost teacher + OOF (5 folds, CPU)...


XGB OOF: 0it [00:00, ?it/s]

    done in 1.3s  |  test AUC = 0.9229

Teacher diagnostics:
  TabICL      train=0.9138  test=0.9224  LL=0.2887  Brier=0.0913
  XGBoost     train=0.9153  test=0.9229  LL=0.2868  Brier=0.0909

Teacher agreement (Spearman): ρ = 0.9781

[4/4] EBM students  (3 configs × 5 seeds × 2 folds)


...

performance:   0%|                                                                                            …

  EBM / binary:   0%|                                                                                         …

  EBM / TabICL probs:   0%|                                                                                   …

  EBM / XGB probs:   0%|                                                                                      …

    performance done in 47.7s

COMBINED  [USE_CALIBRATED=False, TABICL_EST=4, TABICL_OOF_SPL=3, N_SEEDS=5, EBM_MAX_ROUNDS=50]
             model    AUC  AUC_std  LogLoss  Brier    ECE  disagreement  ambiguity  explanation_corr
  TabICL (teacher) 0.9224   0.0000   0.2887 0.0913 0.0181           NaN        NaN               NaN
 XGBoost (teacher) 0.9229   0.0000   0.2868 0.0909 0.0164           NaN        NaN               NaN
      EBM / binary 0.9065   0.0002   0.3199 0.1019 0.0334        0.0035     0.0000            0.8099
EBM / TabICL probs 0.9042   0.0002   0.3221 0.1029 0.0263        0.0040     0.0000            0.8011
   EBM / XGB probs 0.9056   0.0003   0.3216 0.1024 0.0346        0.0027     0.0000            0.8619


In [9]:
combined

,model,AUC,AUC_std,LogLoss,Brier,ECE,disagreement,ambiguity,explanation_corr
0,TabICL (teacher),0.922365,0.000000,0.288734,0.091317,0.018074,NaN,NaN,NaN
1,XGBoost (teacher),0.922896,0.000000,0.286756,0.090881,0.016378,NaN,NaN,NaN
2,EBM / binary,0.906461,0.000205,0.319945,0.101910,0.033410,0.003467,0.000021,0.809926
3,EBM / TabICL probs,0.904237,0.000200,0.322082,0.102942,0.026338,0.004000,0.000020,0.801118
4,EBM / XGB probs,0.905552,0.000307,0.321554,0.102351,0.034613,0.002693,0.000016,0.861907
